In [1]:
import sys
import time
import torch
import torchvision
from pathlib import Path
import os
import numpy as np
from torch.utils.data import Dataset
from PIL import Image
from visdrone_toolkit import VisDroneDataset
from visdrone_toolkit.utils import collate_fn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn_v2,
    FasterRCNN_ResNet50_FPN_V2_Weights
)
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from torchmetrics.detection import MeanAveragePrecision


C:\Users\Ray\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_img = r"D:\cv\Dataset\VisDrone2019-DET-train\images"
train_annotations =r"D:\cv\Dataset\VisDrone2019-DET-train\annotations"

val_img = r"D:\cv\Dataset\VisDrone2019-DET-val\images"
val_annotations= r"D:\cv\Dataset\VisDrone2019-DET-val\annotations"

test_img = r"D:\cv\Dataset\VisDrone2019-DET-test-dev\images"
test_annotations = r"D:\cv\Dataset\VisDrone2019-DET-test-dev\annotations"





In [3]:
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

torch.set_float32_matmul_precision("high")

weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT

model = fasterrcnn_resnet50_fpn_v2(
    weights=weights
)

device = torch.device("cuda")
model = model.to(device)

In [ ]:
train_dataset = VisDroneDataset(
    image_dir=train_img,
    annotation_dir=train_annotations,
    filter_ignored=True,
    filter_crowd=True,
)



Found 6471 images in D:\cv\Dataset\VisDrone2019-DET-train\images
Found 548 images in D:\cv\Dataset\VisDrone2019-DET-val\images


In [ ]:
in_features = model.roi_heads.box_predictor.cls_score.in_features

model.roi_heads.box_predictor = FastRCNNPredictor(
    in_features,
)
model = model.to(device)

In [6]:
image, target = train_dataset[0]

print("image:", image.shape)

for key, value in target.items():
    print(
        key,
        tuple(value.shape) if torch.is_tensor(value) else type(value),
        value.dtype if torch.is_tensor(value) else ""
    )

image: torch.Size([3, 450, 800])
boxes (82, 4) torch.float32
labels (82,) torch.int64
image_id (1,) torch.int64
area (82,) torch.float32
iscrowd (82,) torch.int64


In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=8,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
    collate_fn=collate_fn,
)


model.transform.min_size = (800,)
model.transform.max_size = 800

model.rpn.pre_nms_top_n_train = 1000
model.rpn.post_nms_top_n_train = 500
model.rpn.pre_nms_top_n_test = 1000
model.rpn.post_nms_top_n_test = 500

model.roi_heads.detections_per_img = 500

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.0025,
    momentum=0.9,
    weight_decay=0.0005
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=450
)

scaler = torch.amp.GradScaler("cuda")

EPOCHS = 100

In [8]:
for epoch in range(EPOCHS):

    model.train()
    epoch_loss = 0.0

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}",
        unit="batch",
        dynamic_ncols=True
    )

    for images, targets in progress_bar:

        images = [
            image.to(device, non_blocking=True)
            for image in images
        ]

        targets = [
            {
                key: value.to(device, non_blocking=True)
                if torch.is_tensor(value) else value
                for key, value in target.items()
            }
            for target in targets
        ]

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            loss_dict = model(images, targets)
            loss = sum(loss_dict.values())

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}",
            cls=f"{loss_dict['loss_classifier'].item():.3f}",
            box=f"{loss_dict['loss_box_reg'].item():.3f}",
            obj=f"{loss_dict['loss_objectness'].item():.3f}",
            rpn=f"{loss_dict['loss_rpn_box_reg'].item():.3f}"
        )

    epoch_loss /= len(train_loader)

    scheduler.step()

    print(
        f"Epoch [{epoch + 1}/{EPOCHS}] "
        f"Train Loss: {epoch_loss:.4f} | "
        f"LR: {optimizer.param_groups[0]['lr']:.7f}"
    )

print("\nTraining finished.")

torch.save(
    model.state_dict(),
    "fasterrcnn_visdrone_100ep.pth"
)

print("Model saved: fasterrcnn_visdrone_100ep.pth")

Epoch 1/100: 100%|██████████| 1618/1618 [06:00<00:00,  4.49batch/s, box=0.331, cls=0.287, loss=0.9800, obj=0.104, rpn=0.257]


Epoch [1/100] Train Loss: 1.1144 | LR: 0.0025000


Epoch 2/100: 100%|██████████| 1618/1618 [03:52<00:00,  6.94batch/s, box=0.428, cls=0.435, loss=1.2900, obj=0.158, rpn=0.269]


Epoch [2/100] Train Loss: 0.9346 | LR: 0.0024999


Epoch 3/100: 100%|██████████| 1618/1618 [03:54<00:00,  6.91batch/s, box=0.299, cls=0.276, loss=0.7433, obj=0.067, rpn=0.101]


...

Epoch [98/100] Train Loss: 0.3809 | LR: 0.0022187


Epoch 99/100: 100%|██████████| 1618/1618 [03:55<00:00,  6.86batch/s, box=0.181, cls=0.083, loss=0.4166, obj=0.072, rpn=0.081]


Epoch [99/100] Train Loss: 0.3827 | LR: 0.0022131


Epoch 100/100: 100%|██████████| 1618/1618 [03:56<00:00,  6.85batch/s, box=0.115, cls=0.050, loss=0.2239, obj=0.021, rpn=0.037]


Epoch [100/100] Train Loss: 0.3813 | LR: 0.0022076

Training finished.
Model saved: fasterrcnn_visdrone_100ep.pth


393минуты